# 41 - Preparar dados para modelagem e comparar normalizacoes

Este notebook repete o fluxo do notebook 40: carrega os Parquets intermediarios, seleciona descricoes, remove nulos e vazios, normaliza texto, consolida duplicatas preservando frequencia, agrega metadados e valida o resultado.

Tambem cria colunas experimentais sem substituir `descricao_normalizada`: uma padroniza abreviacoes e unidades; outra uniformiza a separacao entre numeros e unidades; e uma terceira mascara valores numericos. As colunas experimentais sao alternativas de entrada para os proximos experimentos de TF-IDF e podem conter descricoes repetidas, por isso nao alteram a chave de consolidacao nem a frequencia. A gravacao final usa um arquivo separado e falha se ele ja existir.

Opcionalmente, remove uma lista conservadora de stop words das quatro representações antes da consolidação. A nova chave pode unir descrições antes distintas; frequências e metadados são recalculados a partir dos itens restantes. Descrições normalizadas que ficam vazias são excluídas e contabilizadas. Não há lematização.

Também permite gerar uma base amostral com tamanho máximo configurável e seed 42. A seleção ocorre após a preparação e a validação da base completa, usando amostragem uniforme sem reposição, como nos notebooks da série 70.

In [20]:
from __future__ import annotations

import sys
import re
import unicodedata
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

## Configuracao

Defina `MANTER_DUPLICATAS = True` para gerar uma linha por item valido, sem consolidar descricoes normalizadas. Com `False`, o fluxo consolida descricoes e registra a frequencia, como antes. Cada modo usa um nome de saida proprio.

As regras experimentais abaixo sao deliberadamente conservadoras. Revise os exemplos e compare cada coluna em experimentos separados antes de escolher uma representacao para o modelo.

Defina `REMOVER_STOP_WORDS = True` para gerar a base **`2025_NCMs_filtrados_semstopwords`** (modo consolidado). Com `MANTER_DUPLICATAS = True`, o nome recebe `-com-duplicatas`. Com a remoção desabilitada, o modo consolidado mantém o nome convencional; o modo com duplicatas usa `-com-duplicatas`.

`STOP_WORDS` é uma lista local e editável, com artigos e conectivos. Por padrão, preserva `com`, `sem`, `nao` e `para`, assim como marcas, números e unidades que não constem da lista. A remoção considera tokens completos após a normalização; não remove trechos dentro de palavras. Ao editar a lista, use uma palavra por entrada. As configurações permanecem neste notebook, sem downloads ou novas dependências.

Defina `GERAR_AMOSTRA = True` para salvar somente uma amostra da base preparada. `MAX_DESCRIPTIONS = 10_000` é o limite inicial e `SAMPLE_RANDOM_STATE = 42` controla a reprodução. Quando houver menos linhas disponíveis, todas serão mantidas. A opção começa desabilitada para preservar a geração da base completa.

Com duplicatas consolidadas, a unidade amostral é a descrição normalizada única, sem ponderação por `frequencia`, como na série 70. Com duplicatas mantidas, a unidade amostral é a linha de item; descrições podem se repetir. A amostra não é estratificada por mês ou NCM. Todas as colunas e frequências das linhas selecionadas são preservadas, sem recontagem ou extrapolação para o dataset completo.

A seleção usa a base ordenada por `descricao_normalizada`; o resultado também é ordenado por essa coluna. O nome recebe `-amostra10000-seed42` com os parâmetros iniciais, em uma pasta própria. O número no nome é o limite solicitado; o resumo mostra o tamanho efetivo.

In [21]:
DATASET_NAME = "2025_NCMs_filtrados"
MANTER_DUPLICATAS = False
REMOVER_STOP_WORDS = True
STOP_WORDS = {
    "a", "as", "o", "os", "de", "da", "das", "do", "dos", "e", "em",
    "no", "na", "nos", "nas", "ao", "aos", "um", "uma", "uns", "umas",
}
STOP_WORDS_EXAMPLE_COUNT = 20

GERAR_AMOSTRA = True
MAX_DESCRIPTIONS = 10_000
SAMPLE_RANDOM_STATE = 42

DESCRIPTION_COLUMNS = [
    "descricao_normalizada",
    "descricao_abreviacoes_unidades",
    "descricao_numero_unidade_padronizada",
    "descricao_numeros_mascarados",
]

if REMOVER_STOP_WORDS:
    EXPERIMENT_DATASET_NAME = f"{DATASET_NAME}_semstopwords"
    if MANTER_DUPLICATAS:
        EXPERIMENT_DATASET_NAME += "-com-duplicatas"
else:
    duplicate_suffix = "com-duplicatas" if MANTER_DUPLICATAS else "sem-duplicatas"
    EXPERIMENT_DATASET_NAME = f"{DATASET_NAME}-{duplicate_suffix}"
if GERAR_AMOSTRA:
    if isinstance(MAX_DESCRIPTIONS, bool) or not isinstance(MAX_DESCRIPTIONS, int) or MAX_DESCRIPTIONS < 1:
        raise ValueError("MAX_DESCRIPTIONS deve ser um inteiro positivo quando GERAR_AMOSTRA=True.")
    EXPERIMENT_DATASET_NAME += f"-amostra{MAX_DESCRIPTIONS}-seed{SAMPLE_RANDOM_STATE}"

FREQUENCY_COLUMN = "frequencia"
NUMERIC_SEQUENCE_PATTERN = r"\d{6,}" # remover sequencias de dígitos das descrições

PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / DATASET_NAME
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / EXPERIMENT_DATASET_NAME

# Mapeamentos testados na coluna descricao_abreviacoes_unidades.
ABBREVIATION_MAP = {
    "und": "un",
    "unidade": "un",
    "unidades": "un",
    "quilo": "kg",
    "quilos": "kg",
    "quilograma": "kg",
    "quilogramas": "kg",
    "grama": "g",
    "gramas": "g",
    "litro": "l",
    "litros": "l",
    "lt": "l",
    "mililitro": "ml",
    "mililitros": "ml",
    "pct": "pacote",
    "c": "com",
    "s": "sem",
}

INTERIM_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/interim/2025_NCMs_filtrados'),
 WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/processed/2025_NCMs_filtrados_semstopwords-amostra10000-seed42'))

## Funcoes de preparacao

In [22]:
def normalize_column_name(column_name: str) -> str:
    """Normalize a column name enough to detect expected fields."""
    text = unicodedata.normalize("NFKD", str(column_name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_column(columns: pd.Index, required_tokens: tuple[str, ...]) -> str:
    """Find the first column whose normalized name contains all required tokens."""
    for column in columns:
        normalized = normalize_column_name(column)
        if all(token in normalized for token in required_tokens):
            return column
    raise KeyError(f"Nao encontrei coluna com tokens {required_tokens}. Colunas: {list(columns)}")


def remove_long_numeric_sequences(value: object) -> str:
    """Remove digit sequences with six or more digits from a description."""
    if pd.isna(value):
        return ""

    text = re.sub(NUMERIC_SEQUENCE_PATTERN, " ", str(value))
    return re.sub(r"\s+", " ", text).strip()


def normalize_description(value: object) -> str:
    """Normalize product descriptions for TF-IDF and duplicate consolidation."""
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_month(period: str) -> str:
    """Convert YYYY-MM to YYYYMM for artifact names."""
    return period.replace("-", "")


def most_frequent_by_description(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Return the most common value per normalized description."""
    valid = frame[["descricao_normalizada", value_column]].dropna(subset=[value_column]).copy()
    valid = valid[valid[value_column].astype(str).str.strip() != ""]
    if valid.empty:
        return pd.DataFrame(columns=["descricao_normalizada", output_column])

    weighted = (
        valid.groupby(["descricao_normalizada", value_column], as_index=False)
        .size()
        .sort_values(["descricao_normalizada", "size"], ascending=[True, False])
    )
    return weighted.drop_duplicates("descricao_normalizada")[["descricao_normalizada", value_column]].rename(
        columns={value_column: output_column}
    )

In [23]:
def normalize_abbreviations_and_units(value: object) -> str:
    """Standardize a small, explicit set of abbreviations and unit names."""
    if pd.isna(value):
        return ""

    # Expand slash abbreviations before punctuation normalization removes `/`.
    raw_text = re.sub(r"(?i)\bc\s*/", "com ", str(value))
    raw_text = re.sub(r"(?i)\bs\s*/", "sem ", raw_text)
    text = normalize_description(raw_text)
    for source, replacement in ABBREVIATION_MAP.items():
        text = re.sub(rf"\b{re.escape(source)}\b", replacement, text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_number_unit_spacing(value: object) -> str:
    """Make number/unit spacing consistent while retaining numeric values."""
    text = normalize_description(value)
    # `500ml`, `500 ml` -> `500 ml`; same for common units.
    text = re.sub(r"(?<=\d)(kg|mg|g|ml|l|un)\b", r" \1", text)
    return re.sub(r"\s+", " ", text).strip()


def mask_numeric_values(value: object) -> str:
    """Replace numeric values with NUMERO while retaining unit tokens."""
    text = normalize_number_unit_spacing(value)
    text = re.sub(r"\b\d+(?:\s+\d+)*\b", " numero ", text)
    return re.sub(r"\s+", " ", text).strip()



def remove_stop_words(value: object, stop_words: set[str]) -> str:
    """Remove whole tokens from an already normalized description."""
    if pd.isna(value):
        return ""
    return " ".join(token for token in str(value).split() if token not in stop_words)


def sample_modeling_items(
    frame: pd.DataFrame, max_descriptions: int, random_state: int
) -> pd.DataFrame:
    """Sample prepared rows uniformly, preserving metadata and original frequencies."""
    if isinstance(max_descriptions, bool) or not isinstance(max_descriptions, int) or max_descriptions < 1:
        raise ValueError("O tamanho máximo da amostra deve ser um inteiro positivo.")
    ordered = frame.sort_values("descricao_normalizada", kind="stable").reset_index(drop=True)
    sampled = ordered.sample(n=min(max_descriptions, len(ordered)), random_state=random_state, replace=False)
    return sampled.sort_values("descricao_normalizada", kind="stable").reset_index(drop=True)

## Testes rapidos da normalizacao

In [24]:
normalization_examples = {
    " ARROZ TIPO 1 5KG ": "arroz tipo 1 5kg",
    "CAFE TORRADO 500g": "cafe torrado 500g",
    "LEITE UHT 1L": "leite uht 1l",
    "ACUCAR-CRISTAL 2 KG": "acucar cristal 2 kg",
}

for raw, expected in normalization_examples.items():
    normalized = normalize_description(raw)
    assert normalized == expected, (raw, normalized, expected)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

informative_examples = ["QUEIJO MUCARELA, KG", "Couve Flor (Hortifruti) Ceasa", "PIMENTAO VERMELHA CEASA"]
for raw in informative_examples:
    normalized = normalize_description(raw)
    assert len(normalized) >= 3, (raw, normalized)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

duplicate_check = pd.DataFrame(
    {
        "descricao_original": ["Arroz Tipo 1", " arroz tipo 1 ", "ARROZ TIPO 1"],
    }
)
duplicate_check["descricao_normalizada"] = duplicate_check["descricao_original"].map(normalize_description)
duplicate_frequency = duplicate_check.groupby("descricao_normalizada").size().iloc[0]
assert duplicate_frequency == len(duplicate_check)

numeric_sequence_examples = {
    "ARROZ COD123456789 TIPO 1": "ARROZ COD TIPO 1",
    "LEITE 12345 1L": "LEITE 12345 1L",
    "ITEM 123456 E LOTE 987654321": "ITEM E LOTE",
}
for raw, expected in numeric_sequence_examples.items():
    actual = remove_long_numeric_sequences(raw)
    assert actual == expected, (raw, actual, expected)

sequence_check = pd.DataFrame({"descricao_original": ["ARROZ COD123456 TIPO 1"]})
sequence_check["descricao_original"] = sequence_check["descricao_original"].map(
    remove_long_numeric_sequences
)
assert len(sequence_check) == 1
assert sequence_check.loc[0, "descricao_original"] == "ARROZ COD TIPO 1"


# Verificações independentes da opção habilitada para a execução principal.
example_stop_words = {"de", "da", "e"}  # Fixture fixa; a lista configurável pode ser editada.
assert {normalize_description(word) for word in STOP_WORDS}.isdisjoint({"com", "sem", "nao", "para", "kg", "g", "l", "ml", "un"})
stopword_examples_check = {
    "CAFE DE CEVADA": "cafe cevada",
    "LEITE SEM LACTOSE COM CALCIO": "leite sem lactose com calcio",
    "NAO CONTEM ACUCAR": "nao contem acucar",
    "CERA PARA MADEIRA 500 ML": "cera para madeira 500 ml",
    "DETERGENTE OMO 1 L": "detergente omo 1 l",
    "DE DA E": "",
    "": "",
}
for raw, expected in stopword_examples_check.items():
    normalized = normalize_description(raw)
    actual = remove_stop_words(normalized, example_stop_words)
    assert actual == expected, (raw, actual, expected)
assert remove_stop_words(None, example_stop_words) == ""
assert remove_stop_words("cafe de cevada", set()) == "cafe de cevada"
assert remove_stop_words("cafe de cevada", example_stop_words) == remove_stop_words(
    "cafe cevada", example_stop_words
)

# A amostragem mantém as frequências e reproduz a seleção da série 70.
sampling_check = pd.DataFrame({
    "descricao_normalizada": ["leite", "arroz", "cafe", "feijao", "banana"],
    "frequencia": [100, 1, 3, 2, 5],
    "primeiro_mes": ["2025-01"] * 5,
})
sampling_result = sample_modeling_items(sampling_check, 3, 42)
sampling_expected = (
    sampling_check.sort_values("descricao_normalizada").reset_index(drop=True)
    .sample(n=3, random_state=42)
    .sort_values("descricao_normalizada").reset_index(drop=True)
)
pd.testing.assert_frame_equal(sampling_result, sampling_expected)
pd.testing.assert_frame_equal(sampling_result, sample_modeling_items(sampling_check, 3, 42))
assert sampling_result["descricao_normalizada"].is_unique
assert len(sample_modeling_items(sampling_check, 10_000, 42)) == len(sampling_check)

## Variantes experimentais de texto

`descricao_normalizada` permanece como baseline. As variantes abaixo usam regras deterministicas e deixam quantidades disponiveis em colunas alternativas: abreviacoes/unidades padronizadas, espacamento numero-unidade canonico e numeros substituidos por `numero`. `c` e `s` sao expandidos apenas como tokens isolados, portanto inspecione exemplos reais antes de interpretar o impacto.

Quando `REMOVER_STOP_WORDS` está habilitada, a mesma lista é aplicada a todas essas representações após sua geração, antes de agrupar os itens por `descricao_normalizada`.

In [25]:
variant_examples = {
    "ARROZ 1KG": ("arroz 1kg", "arroz 1 kg", "arroz numero kg"),
    "ARROZ 1 KG": ("arroz 1 kg", "arroz 1 kg", "arroz numero kg"),
    "LEITE 500ML": ("leite 500ml", "leite 500 ml", "leite numero ml"),
    "LEITE 500 ML": ("leite 500 ml", "leite 500 ml", "leite numero ml"),
    "CAFE 250 GRAMAS": ("cafe 250 gramas", "cafe 250 gramas", "cafe numero gramas"),
}
for raw, expected in variant_examples.items():
    actual = (
        normalize_description(raw),
        normalize_number_unit_spacing(raw),
        mask_numeric_values(raw),
    )
    assert actual == expected, (raw, actual, expected)

assert normalize_abbreviations_and_units("LEITE 1 LITRO") == "leite 1 l"
assert normalize_abbreviations_and_units("SABAO 2 UND") == "sabao 2 un"
assert normalize_abbreviations_and_units("CAFE C/ ACUCAR") == "cafe com acucar"



## Carregar Parquets intermediarios

In [26]:
parquet_files = sorted(INTERIM_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {INTERIM_DIR}")

frames = []
for parquet_file in parquet_files:
    frame = pd.read_parquet(parquet_file)
    frame["competencia"] = parquet_file.stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Arquivos carregados: {len(parquet_files)}")
print(f"Linhas intermediarias: {len(df):,}")
df.head()

Arquivos carregados: 12
Linhas intermediarias: 868,292


,CHAVE DE ACESSO,MODELO,SÉRIE,NÚMERO,NATUREZA DA OPERAÇÃO,DATA EMISSÃO,EVENTO MAIS RECENTE,DATA/HORA EVENTO MAIS RECENTE,CPF/CNPJ Emitente,RAZÃO SOCIAL EMITENTE,...,NÚMERO PRODUTO,DESCRIÇÃO DO PRODUTO/SERVIÇO,CÓDIGO NCM/SH,NCM/SH (TIPO DE PRODUTO),CFOP,QUANTIDADE,UNIDADE,VALOR UNITÁRIO,VALOR TOTAL,competencia
0,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,2.0,ACHOCOLATADO 400G,18069000.0,Outros chocolates e preparações alimentícias c...,5102.0,"110,00",UNIDAD,"5,80","638,00",2025-01
1,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,4.0,"APRESUNTADO, TIPO CARNE SUINA",16010000.0,"Enchidos e produtos semelhantes, de carne, de ...",5102.0,"40,00",KG,"17,50","700,00",2025-01
2,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,5.0,BARRAS CEREAIS FRUTAS SECAS E MEL EMB 24 UND,17049090.0,"Outros produtos de confeitaria, sem cacau",5102.0,"50,00",EMB,"20,00","1000,00",2025-01
3,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,6.0,BISCOITO DOCE COM RECHEIO PCT 78G,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante",5102.0,"39,00",PCT,"1,78","69,42",2025-01
4,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,7.0,BISCOITO MAIZENA 400G,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante",5102.0,"150,00",PCT,"3,85","577,50",2025-01


## Selecionar descricoes

In [27]:
description_col = find_column(df.columns, ("DESCRI", "PRODUTO"))

support_columns = {
    "codigo_ncm": None,
    "ncm_tipo": None,
}

try:
    support_columns["codigo_ncm"] = find_column(df.columns, ("NCM", "SH"))
except KeyError:
    pass

try:
    support_columns["ncm_tipo"] = find_column(df.columns, ("TIPO", "PRODUTO"))
except KeyError:
    pass

selected_columns = [description_col, "competencia"]
selected_columns.extend(column for column in support_columns.values() if column is not None)

rename_columns = {description_col: "descricao_original"}
if support_columns["codigo_ncm"] is not None:
    rename_columns[support_columns["codigo_ncm"]] = "codigo_ncm"
if support_columns["ncm_tipo"] is not None:
    rename_columns[support_columns["ncm_tipo"]] = "ncm_tipo"

df_selected = df[selected_columns].rename(columns=rename_columns)

df_selected.head()

,descricao_original,competencia,codigo_ncm,ncm_tipo
0,ACHOCOLATADO 400G,2025-01,18069000.0,Outros chocolates e preparações alimentícias c...
1,"APRESUNTADO, TIPO CARNE SUINA",2025-01,16010000.0,"Enchidos e produtos semelhantes, de carne, de ..."
2,BARRAS CEREAIS FRUTAS SECAS E MEL EMB 24 UND,2025-01,17049090.0,"Outros produtos de confeitaria, sem cacau"
3,BISCOITO DOCE COM RECHEIO PCT 78G,2025-01,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante"
4,BISCOITO MAIZENA 400G,2025-01,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante"


## Remover sequencias numericas longas, normalizar e consolidar duplicatas

Cadeias com seis ou mais digitos consecutivos sao removidas do texto da descricao antes da normalizacao. A linha e preservada quando restam outros termos; descricoes que ficam vazias continuam sujeitas ao filtro existente de descricoes invalidas.

In [28]:
input_rows = len(df_selected)

df_valid = df_selected.dropna(subset=["descricao_original"]).copy()
df_valid["descricao_original"] = df_valid["descricao_original"].astype(str).str.strip()
df_valid = df_valid[df_valid["descricao_original"] != ""].copy()

numeric_sequences_removed = int(
    df_valid["descricao_original"].str.count(NUMERIC_SEQUENCE_PATTERN).sum()
)
df_valid["descricao_original"] = df_valid["descricao_original"].map(
    remove_long_numeric_sequences
)

df_valid["descricao_normalizada"] = df_valid["descricao_original"].map(normalize_description)
df_valid = df_valid[df_valid["descricao_normalizada"] != ""].copy()

# Variantes para comparar em experimentos; a consolidacao segue o baseline.
df_valid["descricao_abreviacoes_unidades"] = df_valid["descricao_original"].map(
    normalize_abbreviations_and_units
)
df_valid["descricao_numero_unidade_padronizada"] = df_valid["descricao_original"].map(
    normalize_number_unit_spacing
)
df_valid["descricao_numeros_mascarados"] = df_valid["descricao_original"].map(mask_numeric_values)

valid_rows_before_stopwords = len(df_valid)

print(f"Linhas com coluna de descricao: {input_rows:,}")
print(f"Linhas com descricao valida antes de stop words: {valid_rows_before_stopwords:,}")
print(f"Sequencias numericas removidas: {numeric_sequences_removed:,}")
print(f"Linhas removidas por nulos/vazios: {input_rows - valid_rows_before_stopwords:,}")

Linhas com coluna de descricao: 868,292
Linhas com descricao valida antes de stop words: 868,288
Sequencias numericas removidas: 28,213
Linhas removidas por nulos/vazios: 4


## Remoção opcional de stop words e comparação antes/depois

A remoção acontece antes de consolidar duplicatas e não altera `descricao_original`. Por exemplo, `cafe de cevada` e `cafe cevada` passam a compartilhar uma chave; suas ocorrências e competências são agregadas juntas. Um item só é excluído se a representação principal ficar vazia. Variantes alternativas vazias são contabilizadas na tabela para inspeção.

As tabelas mostram alterações nas quatro representações e exemplos distintos da representação principal. O total de itens excluídos por stop words é separado das exclusões anteriores por nulos ou normalização.


In [29]:
unique_descriptions_before_stopwords = df_valid["descricao_normalizada"].nunique()
stopword_changed_items = pd.Series(False, index=df_valid.index)
stopword_empty_rows = 0
stopword_summary_rows = []
stopword_examples = pd.DataFrame(columns=["descricao_original", "antes", "depois"])

if REMOVER_STOP_WORDS:
    normalized_stop_words = {normalize_description(word) for word in STOP_WORDS}
    if any(not word or len(word.split()) != 1 for word in normalized_stop_words):
        raise ValueError("Cada entrada de STOP_WORDS deve conter exatamente um token após normalizar.")

    for column in DESCRIPTION_COLUMNS:
        before = df_valid[column].copy()
        after = before.map(lambda text: remove_stop_words(text, normalized_stop_words))
        changed = before.ne(after)
        stopword_changed_items |= changed
        stopword_summary_rows.append({
            "representacao": column,
            "itens_alterados": int(changed.sum()),
            "tokens_removidos": int((before.str.split().str.len() - after.str.split().str.len()).sum()),
            "itens_com_texto_vazio": int(after.eq("").sum()),
        })
        df_valid[column] = after
        if column == "descricao_normalizada":
            stopword_examples = (
                pd.DataFrame({"descricao_original": df_valid["descricao_original"], "antes": before, "depois": after})
                .loc[changed].drop_duplicates("antes").head(STOP_WORDS_EXAMPLE_COUNT)
            )

    stopword_empty_rows = int(df_valid["descricao_normalizada"].eq("").sum())
    df_valid = df_valid.loc[df_valid["descricao_normalizada"].ne("")].copy()

valid_rows = len(df_valid)
unique_descriptions_after_stopwords = df_valid["descricao_normalizada"].nunique()
print(f"Remoção de stop words habilitada: {REMOVER_STOP_WORDS}")
print(f"Itens alterados em pelo menos uma representação: {int(stopword_changed_items.sum()):,}")
print(f"Itens excluídos por descrição vazia após stop words: {stopword_empty_rows:,}")
print(f"Itens restantes: {valid_rows:,}")
print(f"Descrições únicas antes/depois: {unique_descriptions_before_stopwords:,} / {unique_descriptions_after_stopwords:,}")
if REMOVER_STOP_WORDS:
    display(pd.DataFrame(stopword_summary_rows))
    display(stopword_examples.reset_index(drop=True))


Remoção de stop words habilitada: True
Itens alterados em pelo menos uma representação: 236,147
Itens excluídos por descrição vazia após stop words: 1
Itens restantes: 868,287
Descrições únicas antes/depois: 149,149 / 146,999


,representacao,itens_alterados,tokens_removidos,itens_com_texto_vazio
0,descricao_normalizada,236147,271051,1
1,descricao_abreviacoes_unidades,236147,271051,1
2,descricao_numero_unidade_padronizada,236147,271051,1
3,descricao_numeros_mascarados,236147,271051,1


,descricao_original,antes,depois
0,BARRAS CEREAIS FRUTAS SECAS E MEL EMB 24 UND,barras cereais frutas secas e mel emb 24 und,barras cereais frutas secas mel emb 24 und
1,FERMENTO QUIMICO EM PÓ LATA 100G,fermento quimico em po lata 100g,fermento quimico po lata 100g
2,MANTEIGA COM SAL VALE DO LEITE 500 G,manteiga com sal vale do leite 500 g,manteiga com sal vale leite 500 g
3,OVO DE GALINHA SAO LUIS CARTELA COM 30 UND,ovo de galinha sao luis cartela com 30 und,ovo galinha sao luis cartela com 30 und
4,PASTILHA SANITARIA EM PEDRA NOVO FRESCOR,pastilha sanitaria em pedra novo frescor,pastilha sanitaria pedra novo frescor
5,RODO PLASTICO DE 60CM COM CABO MADEIRA,rodo plastico de 60cm com cabo madeira,rodo plastico 60cm com cabo madeira
6,"POLPA DE FRUTA TIPO CUPUAÇU, CONGELADA",polpa de fruta tipo cupuacu congelada,polpa fruta tipo cupuacu congelada
7,"VARAL DE CHAO COM ABAS BRANCO 1,44X0,49X0,89 MOR",varal de chao com abas branco 1 44x0 49x0 89 mor,varal chao com abas branco 1 44x0 49x0 89 mor
8,Relógio De Parede Led Digital Grande Termometr...,relogio de parede led digital grande termometr...,relogio parede led digital grande termometro r...
9,COGUMELO EM CONSERVA 100G,cogumelo em conserva 100g,cogumelo conserva 100g


In [30]:
if MANTER_DUPLICATAS:
    df_modeling = df_valid.copy()
    df_modeling[FREQUENCY_COLUMN] = 1
    df_modeling["descricao_original_exemplo"] = df_modeling["descricao_original"]
    df_modeling["descricoes_originais_distintas"] = 1
    df_modeling["primeiro_mes"] = df_modeling["competencia"]
    df_modeling["ultimo_mes"] = df_modeling["competencia"]
    df_modeling["meses_presentes"] = 1
    rename_columns = {
        "codigo_ncm": "codigo_ncm_mais_frequente",
        "ncm_tipo": "ncm_tipo_mais_frequente",
    }
    df_modeling = df_modeling.rename(
        columns={key: value for key, value in rename_columns.items() if key in df_modeling.columns}
    )
    df_modeling = df_modeling.sort_values(
        ["competencia", "descricao_normalizada"], ascending=[True, True]
    ).reset_index(drop=True)
else:
    aggregations = {
        FREQUENCY_COLUMN: ("descricao_original", "size"),
        "descricao_original_exemplo": ("descricao_original", "first"),
        "descricoes_originais_distintas": ("descricao_original", "nunique"),
        "primeiro_mes": ("competencia", "min"),
        "ultimo_mes": ("competencia", "max"),
        "meses_presentes": ("competencia", "nunique"),
        "descricao_abreviacoes_unidades": ("descricao_abreviacoes_unidades", "first"),
        "descricao_numero_unidade_padronizada": ("descricao_numero_unidade_padronizada", "first"),
        "descricao_numeros_mascarados": ("descricao_numeros_mascarados", "first"),
    }

    df_modeling = (
        df_valid.groupby("descricao_normalizada", as_index=False)
        .agg(**aggregations)
        .sort_values([FREQUENCY_COLUMN, "descricao_normalizada"], ascending=[False, True])
        .reset_index(drop=True)
    )

    if "codigo_ncm" in df_valid.columns:
        ncm_mode = most_frequent_by_description(df_valid, "codigo_ncm", "codigo_ncm_mais_frequente")
        df_modeling = df_modeling.merge(ncm_mode, on="descricao_normalizada", how="left")

    if "ncm_tipo" in df_valid.columns:
        ncm_tipo_mode = most_frequent_by_description(df_valid, "ncm_tipo", "ncm_tipo_mais_frequente")
        df_modeling = df_modeling.merge(ncm_tipo_mode, on="descricao_normalizada", how="left")

df_modeling.head(10)


,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,papel a4 br go office 75g 500fl certfscmisto70...,7814,Papel A4 BR Go Office 75g 500fl CertFSCMisto70...,1,2025-01,2025-12,12,papel a4 br go office 75g 500fl certfscmisto70...,papel a4 br go office 75 g 500fl certfscmisto7...,papel a4 br go office numero g 500fl certfscmi...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
1,pilha alcalina go tech aaa lr03 palito 2un,3588,Pilha Alcalina Go Tech AAA LR03 Palito 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aaa lr03 palito 2un,pilha alcalina go tech aaa lr03 palito 2 un,pilha alcalina go tech aaa lr03 palito numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
2,farinha mandioca,3393,FARINHA DE MANDIOCA,11,2025-01,2025-12,12,farinha mandioca,farinha mandioca,farinha mandioca,11062000.0,"Farinhas, sêmolas e pós, de sagu ou das raízes..."
3,papel a4 75g 500fls autopaper fsc credito mist...,3115,PAPEL A4 75G 500FLS AUTOPAPER FSC CREDITO MIST...,1,2025-03,2025-12,10,papel a4 75g 500fls autopaper fsc credito mist...,papel a4 75 g 500fls autopaper fsc credito mis...,papel a4 numero g 500fls autopaper fsc credito...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
4,pilha alcalina go tech aa lr6 pequena 2un,3032,Pilha Alcalina Go Tech AA LR6 Pequena 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aa lr6 pequena 2un,pilha alcalina go tech aa lr6 pequena 2 un,pilha alcalina go tech aa lr6 pequena numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
5,20 litros sem gas agua mineral,2272,20 LITROS SEM GAS - AGUA MINERAL,1,2025-01,2025-12,12,20 l sem gas agua mineral,20 litros sem gas agua mineral,numero litros sem gas agua mineral,22011000.0,"Águas minerais e águas gaseificadas, não adici..."
6,esponja dupla face,2220,ESPONJA DUPLA FACE,6,2025-01,2025-12,12,esponja dupla face,esponja dupla face,esponja dupla face,68053090.0,Outros abrasivos naturais/artificiais em pó/gr...
7,linguica calabresa,2198,LINGUICA CALABRESA,9,2025-01,2025-12,12,linguica calabresa,linguica calabresa,linguica calabresa,16010000.0,"Enchidos e produtos semelhantes, de carne, de ..."
8,pilha alcalina aaa c 2un,2179,PILHA ALCALINA AAA C/2UN.,2,2025-01,2025-12,12,pilha alcalina aaa com 2un,pilha alcalina aaa c 2 un,pilha alcalina aaa c numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
9,pao frances,2121,PAO FRANCES,13,2025-01,2025-12,12,pao frances,pao frances,pao frances,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."


## Validacoes finais

In [31]:
assert valid_rows_before_stopwords == valid_rows + stopword_empty_rows
assert not df_modeling.empty
assert df_modeling["descricao_normalizada"].notna().all()
assert (df_modeling["descricao_normalizada"].str.strip() != "").all()
assert int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows
if not MANTER_DUPLICATAS:
    assert df_modeling["descricao_normalizada"].is_unique
else:
    assert len(df_modeling) == valid_rows
    assert df_modeling[FREQUENCY_COLUMN].eq(1).all()

print(f"Linhas na base: {len(df_modeling):,}")
print(f"Descricoes unicas normalizadas: {df_modeling['descricao_normalizada'].nunique():,}")
print(f"Duplicatas mantidas: {MANTER_DUPLICATAS}")
print(f"Linhas validas consolidadas: {int(df_modeling[FREQUENCY_COLUMN].sum()):,}")
print(f"Frequencia preservada: {int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows}")

Linhas na base: 146,999
Descricoes unicas normalizadas: 146,999
Duplicatas mantidas: False
Linhas validas consolidadas: 868,287
Frequencia preservada: True


## Amostragem aleatória opcional da base preparada

A base completa já foi validada acima. Nesta etapa, selecionamos no máximo `MAX_DESCRIPTIONS` linhas sem reposição, com `SAMPLE_RANDOM_STATE`. No modo consolidado, a seleção é uniforme por descrição única; frequências maiores não aumentam a chance de seleção.

`frequencia` e os metadados continuam descrevendo as ocorrências originais das descrições selecionadas. Por isso, a soma de frequências da amostra pode ser maior que seu número de linhas e menor que o total da base completa. No modo com duplicatas, cada linha selecionada continua com frequência 1. Não há divisão de treino/validação nesta etapa.


In [32]:
modeling_rows_before_sampling = len(df_modeling)
frequency_before_sampling = int(df_modeling[FREQUENCY_COLUMN].sum())

if GERAR_AMOSTRA:
    df_modeling = sample_modeling_items(df_modeling, MAX_DESCRIPTIONS, SAMPLE_RANDOM_STATE)
    assert len(df_modeling) == min(MAX_DESCRIPTIONS, modeling_rows_before_sampling)
else:
    assert len(df_modeling) == modeling_rows_before_sampling

assert not df_modeling.empty
assert int(df_modeling[FREQUENCY_COLUMN].sum()) <= frequency_before_sampling
if not MANTER_DUPLICATAS:
    assert df_modeling["descricao_normalizada"].is_unique
else:
    assert df_modeling[FREQUENCY_COLUMN].eq(1).all()

print(f"Amostragem habilitada: {GERAR_AMOSTRA}")
if GERAR_AMOSTRA:
    print(f"Limite solicitado: {MAX_DESCRIPTIONS:,}; seed: {SAMPLE_RANDOM_STATE}")
print(f"Linhas na base completa: {modeling_rows_before_sampling:,}")
print(f"Linhas selecionadas para saída: {len(df_modeling):,}")
print(f"Ocorrências representadas na saída: {int(df_modeling[FREQUENCY_COLUMN].sum()):,} de {frequency_before_sampling:,}")
print(f"Dataset de saída: {EXPERIMENT_DATASET_NAME}")
display(df_modeling.head())


Amostragem habilitada: True
Limite solicitado: 10,000; seed: 42
Linhas na base completa: 146,999
Linhas selecionadas para saída: 10,000
Ocorrências representadas na saída: 60,396 de 868,287
Dataset de saída: 2025_NCMs_filtrados_semstopwords-amostra10000-seed42


,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,0 25kg adesivo epoxi massa bi componente com b...,1,"0,25KG DE ADESIVO EPOXI MASSA BI COMPONENTE CO...",1,2025-12,2025-12,1,0 25kg adesivo epoxi massa bi componente com b...,0 25 kg adesivo epoxi massa bi componente com ...,numero kg adesivo epoxi massa bi componente co...,35061090.0,Outros produtos de qualquer espécie utilizados...
1,00007 detergente composicao tensoativo anionic...,1,"00007 - DETERGENTE, COMPOSICAO TENSOATIVO ANIO...",1,2025-04,2025-04,1,00007 detergente composicao tensoativo anionic...,00007 detergente composicao tensoativo anionic...,numero detergente composicao tensoativo anioni...,34029039.0,Outras preparações para lavagem (detergentes)
2,0001 petit four,7,0001 - PETIT FOUR,1,2025-01,2025-12,7,0001 petit four,0001 petit four,numero petit four,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."
3,00054 bolo alimenticio sabor maracuja tipo tor...,2,"00054 - BOLO ALIMENTICIO, SABOR MARACUJA, TIPO...",1,2025-04,2025-05,2,00054 bolo alimenticio sabor maracuja tipo tor...,00054 bolo alimenticio sabor maracuja tipo tor...,numero bolo alimenticio sabor maracuja tipo to...,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."
4,0009 saco plastico ksf 1050x2500,1,0009/SACO PLASTICO KSF 1050X2500,1,2025-11,2025-11,1,0009 saco plastico ksf 1050x2500,0009 saco plastico ksf 1050x2500,numero saco plastico ksf 1050x2500,39232910.0,"Sacos, bolsas e cartuchos, de outros plásticos..."


## Salvar base processada

A saida e gravada em `data/processed/` na pasta definida por `EXPERIMENT_DATASET_NAME`, separada da base do notebook 40. Com duplicatas mantidas, cada linha representa um item valido e frequencia vale 1. No modo consolidado, as colunas alternativas representam a primeira descricao de cada grupo baseline.

Para o período anual, com remoção habilitada e duplicatas consolidadas, o Parquet será `data/processed/2025_NCMs_filtrados_semstopwords/2025_NCMs_filtrados_semstopwords-202501-202512.parquet`. O CSV auxiliar mantém o mesmo prefixo. Os dois destinos são verificados antes de iniciar a gravação; arquivos existentes nunca são sobrescritos. Se editar a lista para outro experimento, escolha um novo `EXPERIMENT_DATASET_NAME`.

Com `GERAR_AMOSTRA = True`, esta célula salva apenas as linhas selecionadas. Por exemplo, com stop words removidas, duplicatas consolidadas, limite 10.000 e seed 42, o dataset será `2025_NCMs_filtrados_semstopwords-amostra10000-seed42`. A base completa permanece disponível ao executar novamente com a opção desabilitada; a amostragem não grava uma segunda cópia completa automaticamente.

In [33]:
start_period = compact_month(min(file.stem for file in parquet_files))
end_period = compact_month(max(file.stem for file in parquet_files))

# Esta célula grava o Parquet e o CSV auxiliar. Verifica ambos antes de gravar.
output_stem = f"{EXPERIMENT_DATASET_NAME}-{start_period}-{end_period}"
output_path = PROCESSED_DIR / f"{output_stem}.parquet"
csv_output_path = PROCESSED_DIR / f"{output_stem}.csv"
existing_outputs = [path for path in (output_path, csv_output_path) if path.exists()]
if existing_outputs:
    raise FileExistsError(
        f"Arquivos processados já existem: {existing_outputs}. "
        "Escolha outro EXPERIMENT_DATASET_NAME para preservar as bases existentes."
    )

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
df_modeling.to_parquet(output_path, index=False)
df_modeling.to_csv(csv_output_path, index=False)

output_path

WindowsPath('C:/Users/samuel.simao/Sources/Nofis-Classifier/data/processed/2025_NCMs_filtrados_semstopwords-amostra10000-seed42/2025_NCMs_filtrados_semstopwords-amostra10000-seed42-202501-202512.parquet')

In [34]:
df_modeling.head()

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,0 25kg adesivo epoxi massa bi componente com b...,1,"0,25KG DE ADESIVO EPOXI MASSA BI COMPONENTE CO...",1,2025-12,2025-12,1,0 25kg adesivo epoxi massa bi componente com b...,0 25 kg adesivo epoxi massa bi componente com ...,numero kg adesivo epoxi massa bi componente co...,35061090.0,Outros produtos de qualquer espécie utilizados...
1,00007 detergente composicao tensoativo anionic...,1,"00007 - DETERGENTE, COMPOSICAO TENSOATIVO ANIO...",1,2025-04,2025-04,1,00007 detergente composicao tensoativo anionic...,00007 detergente composicao tensoativo anionic...,numero detergente composicao tensoativo anioni...,34029039.0,Outras preparações para lavagem (detergentes)
2,0001 petit four,7,0001 - PETIT FOUR,1,2025-01,2025-12,7,0001 petit four,0001 petit four,numero petit four,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."
3,00054 bolo alimenticio sabor maracuja tipo tor...,2,"00054 - BOLO ALIMENTICIO, SABOR MARACUJA, TIPO...",1,2025-04,2025-05,2,00054 bolo alimenticio sabor maracuja tipo tor...,00054 bolo alimenticio sabor maracuja tipo tor...,numero bolo alimenticio sabor maracuja tipo to...,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."
4,0009 saco plastico ksf 1050x2500,1,0009/SACO PLASTICO KSF 1050X2500,1,2025-11,2025-11,1,0009 saco plastico ksf 1050x2500,0009 saco plastico ksf 1050x2500,numero saco plastico ksf 1050x2500,39232910.0,"Sacos, bolsas e cartuchos, de outros plásticos..."


## Analise breve do dataset final

In [35]:
analysis_summary = pd.Series(
    {
        "arquivos_origem": len(parquet_files),
        "linhas_originais": len(df),
        "linhas_validas_antes_stopwords": valid_rows_before_stopwords,
        "remover_stop_words": REMOVER_STOP_WORDS,
        "itens_alterados_stopwords": int(stopword_changed_items.sum()),
        "itens_excluidos_stopwords": stopword_empty_rows,
        "descricoes_unicas_antes_stopwords": unique_descriptions_before_stopwords,
        "descricoes_unicas_apos_stopwords": unique_descriptions_after_stopwords,
        "linhas_com_descricao_valida": valid_rows,
        "gerar_amostra": GERAR_AMOSTRA,
        "limite_amostra": MAX_DESCRIPTIONS if GERAR_AMOSTRA else None,
        "seed_amostra": SAMPLE_RANDOM_STATE if GERAR_AMOSTRA else None,
        "linhas_base_completa": modeling_rows_before_sampling,
        "linhas_base_saida": len(df_modeling),
        "descricoes_unicas_normalizadas": df_modeling["descricao_normalizada"].nunique(),
        "frequencia_base_completa": frequency_before_sampling,
        "frequencia_representada_saida": int(df_modeling[FREQUENCY_COLUMN].sum()),
        "frequencia_media_por_descricao": df_modeling[FREQUENCY_COLUMN].mean(),
        "frequencia_mediana_por_descricao": df_modeling[FREQUENCY_COLUMN].median(),
        "maior_frequencia": int(df_modeling[FREQUENCY_COLUMN].max()),
        "periodo_inicial": df_modeling["primeiro_mes"].min(),
        "periodo_final": df_modeling["ultimo_mes"].max(),
    }
)

analysis_summary

arquivos_origem                           12
linhas_originais                      868292
linhas_validas_antes_stopwords        868288
remover_stop_words                      True
itens_alterados_stopwords             236147
itens_excluidos_stopwords                  1
descricoes_unicas_antes_stopwords     149149
descricoes_unicas_apos_stopwords      146999
linhas_com_descricao_valida           868287
gerar_amostra                           True
limite_amostra                         10000
seed_amostra                              42
linhas_base_completa                  146999
linhas_base_saida                      10000
descricoes_unicas_normalizadas         10000
frequencia_base_completa              868287
frequencia_representada_saida          60396
frequencia_media_por_descricao        6.0396
frequencia_mediana_por_descricao         1.0
maior_frequencia                        3588
periodo_inicial                      2025-01
periodo_final                        2025-12
dtype: obj

In [36]:
variant_collision_summary = pd.Series({
    "descricoes_baseline_unicas": df_modeling["descricao_normalizada"].nunique(),
    "unicas_abreviacoes_unidades": df_modeling["descricao_abreviacoes_unidades"].nunique(),
    "unicas_numero_unidade_padronizada": df_modeling["descricao_numero_unidade_padronizada"].nunique(),
    "unicas_numeros_mascarados": df_modeling["descricao_numeros_mascarados"].nunique(),
})
variant_collision_summary

descricoes_baseline_unicas           10000
unicas_abreviacoes_unidades           9996
unicas_numero_unidade_padronizada     9998
unicas_numeros_mascarados             9850
dtype: int64

In [37]:
frequency_bins = pd.cut(
    df_modeling[FREQUENCY_COLUMN],
    bins=[0, 1, 2, 5, 10, 50, 100, float("inf")],
    labels=["1", "2", "3-5", "6-10", "11-50", "51-100", "101+"],
)

frequency_profile = (
    frequency_bins.value_counts(sort=False)
    .rename_axis("faixa_frequencia")
    .reset_index(name="descricoes")
)
frequency_profile["percentual"] = (
    frequency_profile["descricoes"] / len(df_modeling) * 100
).round(2)

frequency_profile

,faixa_frequencia,descricoes,percentual
0,1,5637,56.37
1,2,1511,15.11
2,3-5,1388,13.88
3,6-10,668,6.68
4,11-50,653,6.53
5,51-100,80,0.80
6,101+,63,0.63


In [38]:
top_descriptions = df_modeling[
    ["descricao_normalizada", "descricao_original_exemplo", FREQUENCY_COLUMN, "meses_presentes"]
].head(20)

top_descriptions

,descricao_normalizada,descricao_original_exemplo,frequencia,meses_presentes
0,0 25kg adesivo epoxi massa bi componente com b...,"0,25KG DE ADESIVO EPOXI MASSA BI COMPONENTE CO...",1,1
1,00007 detergente composicao tensoativo anionic...,"00007 - DETERGENTE, COMPOSICAO TENSOATIVO ANIO...",1,1
2,0001 petit four,0001 - PETIT FOUR,7,7
3,00054 bolo alimenticio sabor maracuja tipo tor...,"00054 - BOLO ALIMENTICIO, SABOR MARACUJA, TIPO...",2,2
4,0009 saco plastico ksf 1050x2500,0009/SACO PLASTICO KSF 1050X2500,1,1
5,00099 salgados diversos tipo rissole tamanho p...,"00099 SALGADOS DIVERSOS, TIPO RISSOLE, TAMANHO...",1,1
6,001 126 pilha tam peq aa 1 5v alcalina,"(001 /126) PILHA TAM. PEQ. AA 1,5V ALCALINA",1,1
7,001 229 bateria 3v cr2032,(001 /229) BATERIA 3V CR2032,1,1
8,00110 garrafa termica material aco 941 12 inox...,"00110 - GARRAFA TERMICA, MATERIAL ACO 941,12 I...",2,2
9,00127 luva p procedimento saude nao 2 848 50 c...,00127 - LUVA P/ PROCEDIMENTO DE SAUDE NAO 2.84...,9,2
